In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.formula.api import logit
from scipy.stats import chi2_contingency, chi2

from pathlib import Path
import pickle
import os

#### Project Question
* Does region, payment method, age, or gender have statistical association with customer churn?

In [2]:
# Establishing directory and file paths
project_dir = Path.cwd()
parent_dir = Path.cwd().parent
dataset = f'{parent_dir}/data/customer_churn_clean.csv'
results_dir = f'{parent_dir}/results/'

# Importing dataset
df=pd.read_csv(dataset, index_col=0)
df.head()

,customer_id,gender,region,subscription_plan,payment_method,age,days_since_last_login,customer_service_calls,monthly_spend,churn
0,10001,Male,East,Basic,PayPal,27.0,27,1.0,61.08,0
1,10002,NaN,West,Pro,PayPal,55.0,146,2.0,73.09,0
2,10003,Female,West,Pro,E-Wallet,NaN,147,2.0,116.30,1
3,10004,Female,South,Basic,Credit Card,NaN,34,1.0,0.58,1
4,10005,Male,North,Basic,Crypto,30.0,146,3.0,46.35,1


In [3]:
# Dataset Information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             10000 non-null  int64  
 1   gender                  9039 non-null   str    
 2   region                  10000 non-null  str    
 3   subscription_plan       10000 non-null  str    
 4   payment_method          9300 non-null   str    
 5   age                     8491 non-null   float64
 6   days_since_last_login   10000 non-null  int64  
 7   customer_service_calls  9950 non-null   float64
 8   monthly_spend           9441 non-null   float64
 9   churn                   10000 non-null  int64  
dtypes: float64(3), int64(3), str(4)
memory usage: 781.4 KB


#### Variables for Analysis
* Target Variable: Churn
    - 1 = Churned
    - 0 = Stayed

* Predictor Variables
    - Region
        - Geographic Territory
    - Gender
        - Reported Customer Gender
    - Payment Method
        - Most Frequent Payment Type
    - Age
        - Customer's Age in Years

#### Objectives
- Identify if there's a statistically significant association between target variable and predictor variables
- Quantify the Odds, Odds Ratio, and Probability of statistically significant variables

**_Methods_**
* Chi-Squared ($\chi^2$) Test of Independence
* Logistic Regression between Churn and Age
* Likelihood Ratio using Chi-Squared Test ($\chi^2$)
* Logistic Regression on Churn Asociated Variables

In [10]:
# File saving function
os.makedirs(results_dir,exist_ok=True)
def save_result(path:str,filename:str,data):
    with open(path+filename+'.pkl', 'wb') as to_write:
        pickle.dump(data,to_write)

# Read saved results function
def read_result(path:str,filename:str):
    with open(path+filename+'.pkl', 'rb') as to_read:
        result = pickle.load(to_read)
        return result

# Chi Test (Isolated and Multiple)
def chi_test(columns,data,target,alpha=0.05,results_dir=None,test_name=None):
    rows=[]
    for col in columns:
        observed=pd.crosstab(data[target],data[col])
        res = chi2_contingency(observed)

        rows.append({
            'Model':f'{target} ~ {col}',
            'Statistics':(res.statistic).round(2),
            'ddof':res.dof,
            'P Value':(f'{res.pvalue:.10f}'),
            'Reject':res.pvalue<alpha
        })
        results=pd.DataFrame(rows)

    if results_dir and test_name:
        save_result(results_dir,test_name,results)

# Logistic modeling
def model_logit(formula:str,data):
    return logit(formula,data).fit(disp=False)

# Logistic Summary Table
def model_table(model):
    return model.summary().tables[1]

# Chi-Squared Test for Controlled Variables
def controlled_chi_test(controls,predictor,target,data,alpha=0.05):
    full = model_logit(f'{target} ~ {predictor} + {controls}', data)
    used = data.loc[full.model.data.row_labels]              # identical rows for both models
    reduced = model_logit(f'{target} ~ {controls}', used)
    if int(full.nobs) != int(reduced.nobs):
        raise ValueError('Models were fit on different rows.')

    lr = 2 * (full.llf - reduced.llf)
    dof = int(full.df_model - reduced.df_model)
    p = chi2.sf(lr, dof)
    return pd.DataFrame({
        'N': int(full.nobs), 'LR': round(lr, 2), 'DoF': dof,
        'P Value': p, 'Reject': p < alpha,
    }, index=[predictor])

# Variable Name Cleaning
def clean_name(model):
    return model.str.replace(
        r'C\(([^)]+)\)\[T\.(.*?)\]',
        lambda a: f'{a.group(1).capitalize()} ~ {a.group(2).capitalize()}',
        regex=True
    )

# Logistic Model Controlled Odds
def explain_model(model,ref):
    intercept = model.params.iloc[0]
    params = model.params.drop('Intercept')

    inter = pd.DataFrame({
        'Odds':np.exp(intercept),
        'Churn Probability':np.exp(intercept)/(1+np.exp(intercept)),
        'Odds Ratio':1,
        'Gap':'reference'
    }, index=[f'{ref}'.capitalize()])

    rest = pd.DataFrame({
        'Odds':np.exp(intercept+params),
        'Churn Probability':np.exp(intercept+params)/(1+np.exp(intercept+params)),
        'Odds Ratio':np.exp(params),
        'Gap':np.exp(intercept+params)/(1+np.exp(intercept+params)) - (np.exp(intercept)/(1+np.exp(intercept)))
    })

    return pd.concat([inter,rest]).reset_index().sort_values(by='Churn Probability')

# Predicted Churn Probability on Controlled Variables
def predict_churn_prob(model, data, var='region'):
    rows = []
    for level in sorted(data[var].unique()):
        tmp = data.copy()
        tmp[var] = level                              # change region only for prediction
        rows.append({
            var.title(): level,
            'Adjusted Churn Probability': model.predict(tmp).mean(),
        })
    return (pd.DataFrame(rows)
            .sort_values('Adjusted Churn Probability')
            .reset_index(drop=True))

##### *Conducting Chi-Squared ($\chi^2$) Test of Independence*
>**_Hypothesis Test_**
>
>Variable | $H_0$ | $H_1$
>---|---|---
>Region | Customer churn is independent of region | Customer churn depends on region
>Gender | Customer churn is independent of region | Customer churn depends on region
>Payment Method | Customer churn is independent of region | Customer churrn >depends on region

##### *Conducting Isolated Logistic Regression*

>**_Hypothesis Test_**
>
>$H_0:$ Customer's Age is independent to Churn
>
>$H_1:$ Customer's Age is dependent to Churn

>---
#### _Chi Squared ($\chi^2$) Test_
>Target Variable
>* Churn
>
>Predictor Variables:
>* Region
>* Gender
>* Payment Method

>---
#### _Logistic Regression_
>Target Variable
>* Churn
>
>Predictor Variable:
>* Age

>---

In [5]:
df_test = df[df.columns[1:]].dropna().copy()
df_test

,gender,region,subscription_plan,payment_method,age,days_since_last_login,customer_service_calls,monthly_spend,churn
0,Male,East,Basic,PayPal,27.0,27,1.0,61.08,0
4,Male,North,Basic,Crypto,30.0,146,3.0,46.35,1
5,Male,West,Basic,Crypto,46.0,103,1.0,36.64,1
6,Male,South,Basic,E-Wallet,22.0,62,2.0,0.00,1
7,Non-Binary,East,Pro,E-Wallet,44.0,81,2.0,66.51,1
...,...,...,...,...,...,...,...,...,...
9990,Female,East,Pro,E-Wallet,36.0,55,0.0,123.12,1
9991,Male,South,Basic,Crypto,54.0,76,2.0,98.02,1
9992,Female,East,Plus,Crypto,30.0,123,1.0,168.93,0
9994,Male,West,Basic,Crypto,49.0,159,5.0,69.05,1


In [6]:
# Set variables for chi squared test
cols = df.columns[1:]
controls = ['region','gender','payment_method']

# Set target variable
# test name
# alpha
target = 'churn'
chi2_test_name = 'chi_square_test_results'
confidence_level = 0.05

# Get chi test results
chi_test_result = chi_test(
    controls, # columns
    df_test, # data
    target, # target
    confidence_level, # alpha
    results_dir, # directory for saving
    chi2_test_name # results file name
)
result = read_result(results_dir,chi2_test_name)
result

,Model,Statistics,ddof,P Value,Reject
0,churn ~ region,32.62,3,0.0000003878,True
1,churn ~ gender,0.06,2,0.9709303554,False
2,churn ~ payment_method,5.72,3,0.1263140092,False


In [7]:
# Get isolated logistic regression of churn and age
model = model_logit('churn ~ age',df_test)
summary = model_table(model)
summary

,coef,std err,z,P>|z|,[0.025,0.975]
Intercept,1.1182,0.113,9.865,0.000,0.896,1.340
age,0.0007,0.003,0.230,0.818,-0.005,0.007


_Chi-Squared ($\chi^2$)Test Results_
>Variable | Conclusion | Result
>---|---|---
>Region | Region is statistically significant association with churn | ($\chi^2=31.65\;df=3\;p<0.0001$)
>Gender | Gender have no statistical significant association with churn |($\chi^2=0.07\;df=2\;p>0.05$) 
>Payment Method | Payment Method have no statistical significant association with churn |($\chi^2=0.07\;df=2\;p>0.05$)

_Churn ~ Age Logistic Regression_
* Failed to reject the $H_0$ and conclude that there is no statistically significant association betweem churn and age 

##### _Next Steps_
_Likelihood Ratio using Chi-Squared ($\chi^2$) Test of Independence under Controlled Variables_

>Target Variable:
>- Churn
>
>Predictor Variables:
>- Region
>- Monthly Spend
>- Customer Service Calls
>- Days Since Last login
>- Age

$H_0:$ Region has no statistically significant association with customer churn after controlling for other variables.

$H_1:$ Region has statistically significant association with customer churn after controlling for other variables.

In [8]:
# Specify the columns
cols = ['region','monthly_spend','customer_service_calls','days_since_last_login','age','churn']
controls = 'monthly_spend + customer_service_calls + days_since_last_login + age'
predictor = 'region'
target = 'churn'
controlled_chi_test(
    controls,
    predictor,
    target,
    df[cols].dropna().copy()
)

,N,LR,DoF,P Value,Reject
region,7966,45.24,3,8.232463e-10,True


**_Conclusion_**
* Reject $H_0$ and conclude that region has statistically significant association with customer churn after controlling for other variables

**_Next Steps_**
* Perform an isolated logistic regression between region and churn
    - Gender, Payment Method, and Age are omitted from the model which are non-significant on customer churn

In [12]:
formula = 'churn ~ C(region)'
inf_model = model_logit(formula, df_test)
inf_result = explain_model(inf_model,'East')
inf_result['index'] = clean_name(inf_result['index'])
inf_result

,index,Odds,Churn Probability,Odds Ratio,Gap
0,East,2.480000,0.712644,1.000000,reference
1,Region ~ North,3.044226,0.752734,1.227511,0.04009
3,Region ~ West,3.398950,0.772673,1.370544,0.060029
2,Region ~ South,3.844633,0.793586,1.550255,0.080942


**_Results_**
* All regions shows an unusually high churn rate with a minimum of `71.3%`
* Compared with East (71.3%):
    - North has 4.0 percentage points higher 
    - West is 6.0 points higher
    - South 8.1 points higher in South
* Each step up the ranking adds roughly 2 points